# 7b. Single-Cell QC

## Purpose
Flag low-quality single cells per patient using three criteria applied in cascade:
1. **NaN detection** — cells missing key metadata or feature values
2. **Inherited organoid flags** — cells whose parent organoid was flagged in `7a`
3. **Nucleus outliers** — abnormally small/large nuclei or high mass displacement

Outlier detection (step 3) only runs on cells that passed steps 1 and 2.

This is **step 7b of Stage 4 (image-based profiling)**. It runs once per patient
and depends on `7a.organoid_qc.ipynb` having run first.

## Inputs
- `data/{patient}/image_based_profiles/3.annotated_profiles/sc_anno.parquet`
- `data/{patient}/image_based_profiles/4.qc_profiles/organoid_flagged_outliers.parquet`

## Outputs
- `data/{patient}/image_based_profiles/4.qc_profiles/sc_flagged_outliers.parquet`
  — SC profile with added `Metadata_cqc_*` flag columns

## Notes
- QC flags are additive: a cell can be flagged by multiple criteria simultaneously.
- The `Metadata_cqc_organoid_flagged` column propagates organoid-level flags down
  to all cells belonging to that organoid, linking 7a and 7b outputs.

In [1]:
import os
import pathlib

import pandas as pd
from cosmicqc import find_outliers
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)
# NOTE: previously this line unconditionally overrode bandicoot_check()
# with root_dir, meaning bandicoot was never actually used even when
# mounted. Removed so bandicoot_check()'s own bandicoot-first behavior
# takes effect.

In [2]:
if not in_notebook:
    args = parse_args()
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0037_T1_CQ1"
    image_based_profiles_subparent_name = "image_based_profiles"

## Load profiles and initialize QC flags

QC is applied in three rounds:
1. **NaN detection** (`Metadata_cqc_nan_detected`) — missing ObjectID, volume, or parent
2. **Inherited organoid flags** (`Metadata_cqc_organoid_flagged`, `Metadata_cqc_missing_parent_organoid`)
   — cells whose parent organoid failed QC in 7a, or have no parent organoid at all
3. **Nucleus outliers** — applied only to cells that passed rounds 1 and 2

In [3]:
sc_file = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "3.annotated_profiles/sc_anno.parquet"
)
organoid_file = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "4.qc_profiles/organoid_flagged_outliers.parquet"
)

nucleocentric_annotated_sammed_path = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "3.annotated_profiles/nucleocentric_sammed_anno.parquet"
).resolve()
nucleocentric_annotated_morphem_output_path = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "3.annotated_profiles/nucleocentric_morphem_anno.parquet"
).resolve()
sammed_annotated_sc_profiles_path = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "3.annotated_profiles/sammed_sc_anno.parquet"
).resolve()


output_dir = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "4.qc_profiles"
)
output_dir.mkdir(parents=True, exist_ok=True)

sc_qc_output_path = pathlib.Path(f"{output_dir}/sc_flagged_outliers.parquet").resolve()
sammed_sc_qc_output_path = pathlib.Path(
    f"{output_dir}/sammed_sc_flagged_outliers.parquet"
).resolve()
nucleocentric_sammed_qc_output_path = pathlib.Path(
    f"{output_dir}/nucleocentric_sammed_flagged_outliers.parquet"
).resolve()
nucleocentric_morphem_qc_output_path = pathlib.Path(
    f"{output_dir}/nucleocentric_morphem_flagged_outliers.parquet"
).resolve()

orig_sc_profiles_df = pd.read_parquet(sc_file)
organoid_qc_profiles_df = pd.read_parquet(organoid_file)
# Print the shape and head of the combined organoid profiles DataFrame
print(orig_sc_profiles_df.shape)
orig_sc_profiles_df

(15277, 2662)


,Metadata_Biology_PatientID,Metadata_Biology_PatientTumor,Metadata_Biology_TumorType,Metadata_Experiment_Class,Metadata_Experiment_Dose,Metadata_Experiment_ImageSet,Metadata_Experiment_PlateID,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Treatment,...,Nuclei_NoChannel_VolumeSizeShape_EulerNumber,Nuclei_NoChannel_VolumeSizeShape_Extent,Nuclei_NoChannel_VolumeSizeShape_MaxX,Nuclei_NoChannel_VolumeSizeShape_MaxY,Nuclei_NoChannel_VolumeSizeShape_MaxZ,Nuclei_NoChannel_VolumeSizeShape_MinX,Nuclei_NoChannel_VolumeSizeShape_MinY,Nuclei_NoChannel_VolumeSizeShape_MinZ,Nuclei_NoChannel_VolumeSizeShape_SurfaceArea,Nuclei_NoChannel_VolumeSizeShape_Volume
0,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,1.0,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,...,2,0.251452,1323,1192,15,1222,1072,0,382.807531,457.14
1,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,1.0,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,...,1,0.316001,987,848,15,872,740,0,483.214749,588.71
2,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,1.0,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,...,1,0.481157,1024,945,12,944,855,0,305.222873,415.72
3,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,1.0,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,...,1,0.655688,1077,879,3,997,816,0,46.035314,99.14
4,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,1.0,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,...,1,0.621548,1047,857,12,958,756,1,312.627193,614.58
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15272,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,10.0,NF0037_T1_CQ1__NF0037_T1_CQ1__C8__F4,NF0037_T1_CQ1,MEK1/2 inhibitor,Kinase Inhibitor,Binimetinib,...,1,0.794283,1190,1037,8,1076,885,2,203.532884,825.80
15273,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,10.0,NF0037_T1_CQ1__NF0037_T1_CQ1__C8__F4,NF0037_T1_CQ1,MEK1/2 inhibitor,Kinase Inhibitor,Binimetinib,...,1,0.837504,1025,1153,8,951,1072,6,23.332590,100.40
15274,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,10.0,NF0037_T1_CQ1__NF0037_T1_CQ1__C8__F4,NF0037_T1_CQ1,MEK1/2 inhibitor,Kinase Inhibitor,Binimetinib,...,1,0.632095,941,877,8,903,826,6,11.653911,24.50
15275,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,10.0,NF0037_T1_CQ1__NF0037_T1_CQ1__C8__F4,NF0037_T1_CQ1,MEK1/2 inhibitor,Kinase Inhibitor,Binimetinib,...,1,0.419931,1058,932,8,1002,870,6,16.513708,29.16


In [4]:
sc_profiles_df = orig_sc_profiles_df.copy()
sc_profiles_df["Metadata_cqc_nan_detected"] = (
    sc_profiles_df[
        [
            "Metadata_Object_ObjectID",
            "Metadata_Object_ParentOrganoid",
            "Cell_NoChannel_VolumeSizeShape_Volume",
        ]
    ]
    .isna()
    .any(axis=1)
)
# Print the number of organoids flagged
flagged_count = sc_profiles_df["Metadata_cqc_nan_detected"].sum()
print(f"Number of organoids flagged: {flagged_count}")

sc_profiles_df.head()

Number of organoids flagged: 29


,Metadata_Biology_PatientID,Metadata_Biology_PatientTumor,Metadata_Biology_TumorType,Metadata_Experiment_Class,Metadata_Experiment_Dose,Metadata_Experiment_ImageSet,Metadata_Experiment_PlateID,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Treatment,...,Nuclei_NoChannel_VolumeSizeShape_Extent,Nuclei_NoChannel_VolumeSizeShape_MaxX,Nuclei_NoChannel_VolumeSizeShape_MaxY,Nuclei_NoChannel_VolumeSizeShape_MaxZ,Nuclei_NoChannel_VolumeSizeShape_MinX,Nuclei_NoChannel_VolumeSizeShape_MinY,Nuclei_NoChannel_VolumeSizeShape_MinZ,Nuclei_NoChannel_VolumeSizeShape_SurfaceArea,Nuclei_NoChannel_VolumeSizeShape_Volume,Metadata_cqc_nan_detected
0,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,1.0,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,...,0.251452,1323,1192,15,1222,1072,0,382.807531,457.14,False
1,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,1.0,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,...,0.316001,987,848,15,872,740,0,483.214749,588.71,False
2,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,1.0,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,...,0.481157,1024,945,12,944,855,0,305.222873,415.72,False
3,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,1.0,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,...,0.655688,1077,879,3,997,816,0,46.035314,99.14,False
4,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,1.0,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,...,0.621548,1047,857,12,958,756,1,312.627193,614.58,False


In [5]:
# Round 2: propagate organoid-level QC flags to single cells.
# A cell is flagged if its parent organoid was flagged in 7a.
# We match on (ParentOrganoid, WellFOV) rather than ParentOrganoid alone because
# object IDs are reassigned per-FOV and are not globally unique across the patient.

# Default QC flags
sc_profiles_df["Metadata_cqc_organoid_flagged"] = False
sc_profiles_df["Metadata_cqc_nan_detected"] = (
    sc_profiles_df[
        ["Metadata_Object_ObjectID", "Nuclei_NoChannel_VolumeSizeShape_Volume"]
    ]
    .isna()
    .any(axis=1)
)
sc_profiles_df["Metadata_cqc_missing_parent_organoid"] = (
    sc_profiles_df["Metadata_Object_ParentOrganoid"] == -1
)


organoid_flags_df = organoid_qc_profiles_df[
    ["Metadata_Object_ObjectID", "Metadata_Experiment_WellFOV"]
    + [col for col in organoid_qc_profiles_df.columns if col.startswith("Metadata_cqc")]
]

# Get flagged (object_id, image_set) pairs
flagged_pairs = set(
    organoid_flags_df.loc[
        organoid_flags_df.filter(like="cqc").any(axis=1),
        ["Metadata_Object_ObjectID", "Metadata_Experiment_WellFOV"],
    ].itertuples(index=False, name=None)
)

# Flag SC rows where both parent_organoid & image_set match a flagged organoid
sc_profiles_df["Metadata_cqc_organoid_flagged"] = sc_profiles_df.apply(
    lambda row: (
        (row["Metadata_Object_ParentOrganoid"], row["Metadata_Experiment_WellFOV"])
        in flagged_pairs
    ),
    axis=1,
)

print(sc_profiles_df.shape)
sc_profiles_df.head()

(15277, 2665)


,Metadata_Biology_PatientID,Metadata_Biology_PatientTumor,Metadata_Biology_TumorType,Metadata_Experiment_Class,Metadata_Experiment_Dose,Metadata_Experiment_ImageSet,Metadata_Experiment_PlateID,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Treatment,...,Nuclei_NoChannel_VolumeSizeShape_MaxY,Nuclei_NoChannel_VolumeSizeShape_MaxZ,Nuclei_NoChannel_VolumeSizeShape_MinX,Nuclei_NoChannel_VolumeSizeShape_MinY,Nuclei_NoChannel_VolumeSizeShape_MinZ,Nuclei_NoChannel_VolumeSizeShape_SurfaceArea,Nuclei_NoChannel_VolumeSizeShape_Volume,Metadata_cqc_nan_detected,Metadata_cqc_organoid_flagged,Metadata_cqc_missing_parent_organoid
0,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,1.0,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,...,1192,15,1222,1072,0,382.807531,457.14,False,False,True
1,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,1.0,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,...,848,15,872,740,0,483.214749,588.71,False,False,False
2,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,1.0,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,...,945,12,944,855,0,305.222873,415.72,False,False,False
3,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,1.0,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,...,879,3,997,816,0,46.035314,99.14,False,False,False
4,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,1.0,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,...,857,12,958,756,1,312.627193,614.58,False,False,False


In [6]:
sc_profiles_df["Nuclei_NoChannel_VolumeSizeShape_Volume"].describe()

count    15248.000000
mean       345.432679
std        270.528183
min          5.040000
25%        130.717500
50%        306.560000
75%        485.395000
max       3603.450000
Name: Nuclei_NoChannel_VolumeSizeShape_Volume, dtype: float64

## Detect outlier single-cells using the non-flagged data

We will attempt to detect instances of poor quality segmentations using the nuclei compartment as the base. The conditions we are using are as follows:

1. Abnormally small or large nuclei using `Volume`
2. Abnormally high `mass displacement` in the nuclei for instances of mis-segmentation of background/no longer in-focus

In [7]:
# Set the metadata columns to be used in the QC process
metadata_columns = [x for x in sc_profiles_df.columns if "Metadata" in x]

In [8]:
# Round 3: nucleus-based outlier detection using z-score thresholds.
# Threshold sign: negative = flag below mean, positive = flag above mean.
# Threshold magnitude: number of standard deviations from the mean.
# Only cells that passed rounds 1 and 2 are evaluated here.
# Only process the rows that are not flagged
filtered_plate_df = sc_profiles_df[
    ~(
        sc_profiles_df["Metadata_cqc_nan_detected"]
        | sc_profiles_df["Metadata_cqc_organoid_flagged"]
        | sc_profiles_df["Metadata_cqc_missing_parent_organoid"]
    )
]

# --- Find size based nuclei outliers ---
print("Finding small nuclei outliers...")
small_nuclei_outliers = find_outliers(
    df=filtered_plate_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        "Nuclei_NoChannel_VolumeSizeShape_Volume": -1,  # Detect very small nuclei
    },
)

# Ensure the column exists before assignment
sc_profiles_df["Metadata_cqc_small_nuclei_outlier"] = False
sc_profiles_df.loc[small_nuclei_outliers.index, "Metadata_cqc_small_nuclei_outlier"] = (
    True
)

print("Finding large nuclei outliers...")
large_nuclei_outliers = find_outliers(
    df=filtered_plate_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        "Nuclei_NoChannel_VolumeSizeShape_Volume": 2,  # Detect very large nuclei
    },
)

# Ensure the column exists before assignment
sc_profiles_df["Metadata_cqc_large_nuclei_outlier"] = False
sc_profiles_df.loc[large_nuclei_outliers.index, "Metadata_cqc_large_nuclei_outlier"] = (
    True
)

# --- Find mass displacement based nuclei outliers ---
print("Finding high mass displacement outliers...")
high_mass_displacement_outliers = find_outliers(
    df=filtered_plate_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        "Nuclei_DNA_Intensity_MassDisplacement": 2,  # Detect high mass displacement
    },
)

# Ensure the column exists before assignment
sc_profiles_df["Metadata_cqc_mass_displacement_outlier"] = False
sc_profiles_df.loc[
    high_mass_displacement_outliers.index, "Metadata_cqc_mass_displacement_outlier"
] = True

# Print number of outliers (only in filtered rows)
small_count = filtered_plate_df.index.intersection(small_nuclei_outliers.index).shape[0]
large_count = filtered_plate_df.index.intersection(large_nuclei_outliers.index).shape[0]
high_mass_count = filtered_plate_df.index.intersection(
    high_mass_displacement_outliers.index
).shape[0]

print(f"Small nuclei outliers found: {small_count}")
print(f"Large nuclei outliers found: {large_count}")
print(f"High mass displacement outliers found: {high_mass_count}")

# Save updated plate_df with flag columns included
sc_profiles_df.to_parquet(sc_qc_output_path, index=False)

Finding small nuclei outliers...
Number of outliers: 1233 (18.12%)
Outliers Range:
Nuclei_NoChannel_VolumeSizeShape_Volume Min: 5.340000000000001
Nuclei_NoChannel_VolumeSizeShape_Volume Max: 98.62000000000002
Finding large nuclei outliers...
Number of outliers: 242 (3.56%)
Outliers Range:
Nuclei_NoChannel_VolumeSizeShape_Volume Min: 851.6300000000001
Nuclei_NoChannel_VolumeSizeShape_Volume Max: 2471.6900000000005
Finding high mass displacement outliers...
Number of outliers: 324 (4.76%)
Outliers Range:
Nuclei_DNA_Intensity_MassDisplacement Min: 1.1490061
Nuclei_DNA_Intensity_MassDisplacement Max: 2.9424338
Small nuclei outliers found: 1233
Large nuclei outliers found: 242
High mass displacement outliers found: 324


In [9]:
sc_profiles_df.head()

,Metadata_Biology_PatientID,Metadata_Biology_PatientTumor,Metadata_Biology_TumorType,Metadata_Experiment_Class,Metadata_Experiment_Dose,Metadata_Experiment_ImageSet,Metadata_Experiment_PlateID,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Treatment,...,Nuclei_NoChannel_VolumeSizeShape_MinY,Nuclei_NoChannel_VolumeSizeShape_MinZ,Nuclei_NoChannel_VolumeSizeShape_SurfaceArea,Nuclei_NoChannel_VolumeSizeShape_Volume,Metadata_cqc_nan_detected,Metadata_cqc_organoid_flagged,Metadata_cqc_missing_parent_organoid,Metadata_cqc_small_nuclei_outlier,Metadata_cqc_large_nuclei_outlier,Metadata_cqc_mass_displacement_outlier
0,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,1.0,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,...,1072,0,382.807531,457.14,False,False,True,False,False,False
1,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,1.0,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,...,740,0,483.214749,588.71,False,False,False,False,False,False
2,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,1.0,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,...,855,0,305.222873,415.72,False,False,False,False,False,False
3,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,1.0,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,...,816,0,46.035314,99.14,False,False,False,False,False,False
4,NF0037,NF0037_T1_CQ1,cNF,Small Molecule,1.0,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,...,756,1,312.627193,614.58,False,False,False,False,False,False


### Merge the qc flags to the deep learning-based profiles and save the output
Merge the QC flags back to the original single cell profiles, which will be used in downstream analyses and single cell QC. 
We need to do this beacuase we do not run qc on black-box features. 
Merge on the Metadata_Biology_PatientTumor, Metadata_Experiment_WellFOV
and the Metadata_Object_ObjectID columns, which together uniquely identify each organoid profile row.

In [10]:
# Each deep-learning profile is only added to df_dict (and therefore QC-flag
# propagated below) when this dataset actually produced it -- absent for
# datasets with no deep-learning features (e.g. ZEDProfiler-only), where
# 6.annotation.py never writes any of these 3 files.
df_dict = {}
if nucleocentric_annotated_sammed_path.exists():
    df_dict["nulceocentric_sammed"] = {
        "df": pd.read_parquet(nucleocentric_annotated_sammed_path),
        "qc_output_path": nucleocentric_sammed_qc_output_path,
    }
if nucleocentric_annotated_morphem_output_path.exists():
    df_dict["nucleocentric_chammi"] = {
        "df": pd.read_parquet(nucleocentric_annotated_morphem_output_path),
        "qc_output_path": nucleocentric_morphem_qc_output_path,
    }
if sammed_annotated_sc_profiles_path.exists():
    df_dict["sammed_sc_profiles"] = {
        "df": pd.read_parquet(sammed_annotated_sc_profiles_path),
        "qc_output_path": sammed_sc_qc_output_path,
    }
if not df_dict:
    print(
        "No deep-learning annotated profiles found -- skipping DL QC flag "
        "propagation entirely (this dataset has no deep-learning features)."
    )

In [11]:
# set the merge keys to int for both dataframes to ensure they match
merge_keys = [
    "Metadata_Biology_PatientTumor",
    "Metadata_Experiment_WellFOV",
    "Metadata_Object_ObjectID",
]
qc_keys = [col for col in sc_profiles_df.columns if "Metadata_cqc" in col]

for profile_name in df_dict:
    df = df_dict[profile_name]["df"]
    for key in merge_keys:
        if key not in df.columns:
            raise ValueError(f"Merge key {key} not found in dataframe columns.")
    qc_annotated_df = df.merge(
        sc_profiles_df[qc_keys + merge_keys],
        on=merge_keys,
        how="left",
    )
    if qc_annotated_df.shape[1] == df.shape[1]:
        raise ValueError(
            f"No new columns were added during the merge. Check that the merge keys {merge_keys} are correct and that the qc keys {qc_keys} are present in the sc_profiles_df."
        )
    qc_annotated_df.to_parquet(df_dict[profile_name]["qc_output_path"], index=False)

In [12]:
qc_annotated_df

,Metadata_Biology_PatientTumor,Metadata_Biology_TumorType,Metadata_Experiment_Class,Metadata_Experiment_Dose,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Treatment,Metadata_Experiment_Unit,Metadata_Experiment_ViabilityPercentage,Metadata_Experiment_Well,...,Nuclei_Mito_SAMMed3D_Feature-global96,Nuclei_Mito_SAMMed3D_Feature-global97,Nuclei_Mito_SAMMed3D_Feature-global98,Nuclei_Mito_SAMMed3D_Feature-global99,Metadata_cqc_nan_detected,Metadata_cqc_organoid_flagged,Metadata_cqc_missing_parent_organoid,Metadata_cqc_small_nuclei_outlier,Metadata_cqc_large_nuclei_outlier,Metadata_cqc_mass_displacement_outlier
0,NF0037_T1_CQ1,cNF,Small Molecule,1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,uM,85.253054,B9,...,0.168289,0.021358,0.289452,0.035058,NaN,NaN,<NA>,NaN,NaN,NaN
1,NF0037_T1_CQ1,cNF,Small Molecule,1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,uM,85.253054,B9,...,0.169738,0.021936,0.287371,0.034826,NaN,NaN,<NA>,NaN,NaN,NaN
2,NF0037_T1_CQ1,cNF,Small Molecule,1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,uM,85.253054,B9,...,0.168661,0.022426,0.286912,0.035584,NaN,NaN,<NA>,NaN,NaN,NaN
3,NF0037_T1_CQ1,cNF,Small Molecule,1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,uM,85.253054,B9,...,0.169004,0.021288,0.289327,0.034722,NaN,NaN,<NA>,NaN,NaN,NaN
4,NF0037_T1_CQ1,cNF,Small Molecule,1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,uM,85.253054,B9,...,0.170030,0.024475,0.283027,0.035380,NaN,NaN,<NA>,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14822,NF0037_T1_CQ1,cNF,Small Molecule,10,MEK1/2 inhibitor,Kinase Inhibitor,Binimetinib,uM,99.389180,C8,...,0.168874,0.022579,0.287080,0.037015,NaN,NaN,<NA>,NaN,NaN,NaN
14823,NF0037_T1_CQ1,cNF,Small Molecule,10,MEK1/2 inhibitor,Kinase Inhibitor,Binimetinib,uM,99.389180,C8,...,0.168903,0.022074,0.289039,0.034836,NaN,NaN,<NA>,NaN,NaN,NaN
14824,NF0037_T1_CQ1,cNF,Small Molecule,10,MEK1/2 inhibitor,Kinase Inhibitor,Binimetinib,uM,99.389180,C8,...,0.168823,0.022259,0.288394,0.036162,NaN,NaN,<NA>,NaN,NaN,NaN
14825,NF0037_T1_CQ1,cNF,Small Molecule,10,MEK1/2 inhibitor,Kinase Inhibitor,Binimetinib,uM,99.389180,C8,...,0.168651,0.021968,0.288804,0.036051,NaN,NaN,<NA>,NaN,NaN,NaN
